# RQ2: Does drive type (AWD/FWD/RWD) moderate the relationship between battery capacity and fast-charging speed?

**Research Question:** Does the relationship between battery capacity (kWh) and charging speed (kW) differ significantly by drivetrain configuration?

**Hypothesis:** AWD vehicles have larger batteries requiring higher charging rates, while FWD budget vehicles show weaker coupling between capacity and charging speed.

**Methodology:**
1. Load and clean data; remove outliers (IQR method)
2. Compute group statistics by drive type
3. Pearson correlation per drive type
4. Kruskal-Wallis test for charging speed differences across drive types
5. OLS regression with interaction term: ChargingSpeed ~ BatteryCapacity × DriveType
6. Scatter + regression figure (PDF)
7. Group statistics table (CSV)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import kruskal
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family': 'serif', 'font.size': 11, 'axes.titlesize': 13,
    'axes.labelsize': 12, 'legend.fontsize': 10, 'figure.dpi': 300,
    'axes.spines.top': False, 'axes.spines.right': False,
})

In [ ]:
import os
PATHS = [
    '/kaggle/input/electric-vehicle-market-and-pricing-dataset-2026/ev_market_2026.csv',
    'ev_market_2026.csv',
]
for p in PATHS:
    if os.path.exists(p):
        df = pd.read_csv(p)
        break

df = df.dropna(subset=['battery_capacity_kwh', 'charging_speed_kw', 'drive_type'])

# IQR outlier removal per drive type
clean_dfs = []
for dt in df['drive_type'].unique():
    sub = df[df['drive_type'] == dt].copy()
    for col in ['battery_capacity_kwh', 'charging_speed_kw']:
        Q1, Q3 = sub[col].quantile([0.25, 0.75])
        IQR = Q3 - Q1
        sub = sub[(sub[col] >= Q1 - 1.5*IQR) & (sub[col] <= Q3 + 1.5*IQR)]
    clean_dfs.append(sub)
df_clean = pd.concat(clean_dfs)

DRIVE_TYPES = ['AWD', 'FWD', 'RWD']
COLORS = {'AWD': '#c0392b', 'FWD': '#2980b9', 'RWD': '#27ae60'}
MARKERS = {'AWD': 'o', 'FWD': 's', 'RWD': '^'}
print(f'Clean dataset size: {len(df_clean)}')
print(df_clean['drive_type'].value_counts())

In [ ]:
# ── Statistical tests ─────────────────────────────────────────────────────────
results = []
for dt in DRIVE_TYPES:
    sub = df_clean[df_clean['drive_type'] == dt]
    r, p = stats.pearsonr(sub['battery_capacity_kwh'], sub['charging_speed_kw'])
    results.append({
        'Drive Type': dt, 'N': len(sub),
        'Mean Battery (kWh)': round(sub['battery_capacity_kwh'].mean(), 1),
        'SD Battery (kWh)': round(sub['battery_capacity_kwh'].std(), 1),
        'Mean Charging (kW)': round(sub['charging_speed_kw'].mean(), 1),
        'SD Charging (kW)': round(sub['charging_speed_kw'].std(), 1),
        'Pearson r': round(r, 3), 'p-value': round(p, 4),
    })

stats_df = pd.DataFrame(results)
print(stats_df.to_string(index=False))

groups = [df_clean[df_clean['drive_type'] == dt]['charging_speed_kw'].values for dt in DRIVE_TYPES]
H, p_kw = kruskal(*groups)
print(f'\nKruskal-Wallis H={H:.2f}, p={p_kw:.4f}')

In [ ]:
# ── Figure ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(12, 5), sharey=True)

for ax, dt in zip(axes, DRIVE_TYPES):
    sub = df_clean[df_clean['drive_type'] == dt]
    ax.scatter(sub['battery_capacity_kwh'], sub['charging_speed_kw'],
               color=COLORS[dt], alpha=0.45, s=28, marker=MARKERS[dt])
    slope, intercept, r, _, _ = stats.linregress(sub['battery_capacity_kwh'], sub['charging_speed_kw'])
    x = np.linspace(sub['battery_capacity_kwh'].min(), sub['battery_capacity_kwh'].max(), 100)
    ax.plot(x, intercept + slope * x, color=COLORS[dt], linewidth=2.0, linestyle='--')
    ax.set_title(f'{dt} (n={len(sub)})\nr={r:.2f}', fontsize=11)
    ax.set_xlabel('Battery Capacity (kWh)')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

axes[0].set_ylabel('Charging Speed (kW)')
fig.suptitle(
    'Battery Capacity vs. Fast-Charging Speed by Drivetrain Configuration\n'
    f'(Kruskal-Wallis H={H:.2f}, p={p_kw:.4f})',
    fontsize=13, y=1.01
)
plt.tight_layout()
fig.savefig('RQ2_Battery_Charging_DriveType.pdf', bbox_inches='tight', format='pdf')
plt.show()
print('Figure saved: RQ2_Battery_Charging_DriveType.pdf')

In [ ]:
stats_df.to_csv('RQ2_Summary_Table.csv', index=False)
print('Table saved: RQ2_Summary_Table.csv')
stats_df